# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathibhaShaliniS/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Lane: CTR / Engagement Opportunity Scoring. Method: classification — this is a yes/no question with a label I can define directly from the data (does this page under-capture clicks for its position?), and training-honest-models says exactly that shape calls for Logistic Regression first, then Random Forest, so I'll train both plus a shallow Decision Tree for a version I can actually read and print.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PrathibhaShaliniS/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

usable_tiers = ["page_1", "striking", "page_3_5"]
d = df[df["position_tier"].isin(usable_tiers)].copy()

tier_median = d.groupby("position_tier")["ctr"].transform("median")
d["needs_ctr_review"] = (d["ctr"] < tier_median).astype(int)

print("rows kept:", len(d), "of", len(df))
print(d.groupby("position_tier")["needs_ctr_review"].mean().round(3))


rows kept: 26360 of 30000
position_tier
page_1      0.497
page_3_5    0.492
striking    0.499
Name: needs_ctr_review, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: client-grouped, not random rows. Rows from the same client tend to share templates, writing style, and content strategy — a random row split would let the model see "sister" pages from the same client in both train and test, which overstates how well it'll generalize to a client it's never seen. So the split holds out whole clients: ~20% of clients (6 of 31) go entirely into test, the rest train.

One check worth doing first: with only 31 clients, a single random client split can accidentally land a very unbalanced test set — I checked, and just picking random_state=42 directly gives a test set where 85% of rows need review vs. 48% in train, purely from which clients landed where. So instead of trusting the first split, I searched a small range of seeds for one where train and test label rates are close to each other and to the overall rate — an honest way to avoid an unlucky draw, without picking a seed because it favors my model.

In [4]:
import numpy as np
import pandas as pd

# rebuild d in case section 1 wasn't run in this session
usable_tiers = ["page_1", "striking", "page_3_5"]
d = df[df["position_tier"].isin(usable_tiers)].copy()
tier_median = d.groupby("position_tier")["ctr"].transform("median")
d["needs_ctr_review"] = (d["ctr"] < tier_median).astype(int)

clients = d["client_id"].reset_index(drop=True)
y = d["needs_ctr_review"].to_numpy()
unique_clients = clients.drop_duplicates().to_numpy()

best = None
for seed in range(200):
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(unique_clients)
    n_test = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:n_test])
    test_mask = clients.isin(test_clients).to_numpy()
    gap = abs(y[~test_mask].mean() - y[test_mask].mean())
    if best is None or gap < best[0]:
        best = (gap, seed, test_mask)

_, SPLIT_SEED, test_mask = best
train_mask = ~test_mask

print("chosen seed:", SPLIT_SEED)
print("test clients:", clients[test_mask].nunique(), "of", len(unique_clients))
print("train label rate:", round(y[train_mask].mean(), 3), "| test label rate:", round(y[test_mask].mean(), 3))

chosen seed: 56
test clients: 6 of 31
train label rate: 0.496 | test label rate: 0.496


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same d, same train_mask/test_mask from section 2, same metrics for every row: precision@20/50/100 (ranking metrics — these matter more than accuracy here, since the real use case is "who gets reviewed first"), ROC-AUC, and average precision. The Week-4 baseline score gets evaluated on this exact test split too, using this notebook's own needs_ctr_review label — not its original design target — so it's a genuinely fair, same-data comparison.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

# --- features (ctr and clicks_90d excluded on purpose -- they define the label) ---
d["has_keyword_data"] = d["search_volume"].notna().astype(int)
d["has_word_count"] = d["word_count"].notna().astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "content_age_days", "days_since_last_update",
    "avg_position", "impressions_90d", "has_keyword_data", "has_word_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level",
    "freshness_tier", "word_count_tier", "position_tier",
]

num = d[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
cat = d[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

# --- Week-4 baseline score, recomputed here, evaluated on this same test split ---
d["stale"] = (d["days_since_last_update"] >= 180).astype(int)
d["visible"] = (d["impressions_90d"] >= 500).astype(int)
d["ctr_gap"] = ((d["avg_position"] > 0) & (d["avg_position"] <= 20) & (d["ctr"] < 0.50)).astype(int)
d["baseline_score"] = d["visible"] * d["impressions_90d"] * (d["stale"] + d["ctr_gap"])
baseline_test_scores = d.loc[test_mask, "baseline_score"].to_numpy()

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

def metric_row(y_true, scores, name):
    return {
        "model": name,
        "precision_at_20": round(precision_at_k(y_true, scores, 20), 3),
        "precision_at_50": round(precision_at_k(y_true, scores, 50), 3),
        "precision_at_100": round(precision_at_k(y_true, scores, 100), 3),
        "roc_auc": round(roc_auc_score(y_true, scores), 3),
        "average_precision": round(average_precision_score(y_true, scores), 3),
    }

rows = [metric_row(y_test, baseline_test_scores, "week4_baseline")]

logreg = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
logreg.fit(X_train, y_train)
rows.append(metric_row(y_test, logreg.predict_proba(X_test)[:, 1], "logistic_regression"))

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
rows.append(metric_row(y_test, tree.predict_proba(X_test)[:, 1], "decision_tree_d4"))

rf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=25, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
rows.append(metric_row(y_test, rf_probs, "random_forest"))

results = pd.DataFrame(rows).set_index("model")
print("base rate (test):", round(y_test.mean(), 3))
print(results)


base rate (test): 0.496
                     precision_at_20  precision_at_50  precision_at_100  \
model                                                                     
week4_baseline                  0.15             0.22              0.23   
logistic_regression             0.90             0.84              0.89   
decision_tree_d4                0.65             0.74              0.80   
random_forest                   0.95             0.98              0.98   

                     roc_auc  average_precision  
model                                            
week4_baseline         0.420              0.453  
logistic_regression    0.801              0.796  
decision_tree_d4       0.735              0.717  
random_forest          0.800              0.790  


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What it leans on: impressions_90d and days_with_impressions together account for ~67% of Random Forest's importance. That makes sense but needs a caveat — pages with very few impressions produce noisy CTR ratios (2 impressions and 1 lucky click is a 50% CTR), so the model has partly learned "low volume → predict needs-review" as a blanket rule, which is right most of the time but wrong exactly when a small-sample page got a fluky good or bad ratio. avg_position, content_age_days, word_count, and char_count matter much less individually but still show up — none of them is suspiciously dominant on its own.

Where it's wrong:

False positives (flagged, but actually fine): all 3 worst cases are page_1 rows with only 2–11 impressions total, where one click created a CTR that happened to beat the tier median by chance — the model can't tell a real signal from small-sample noise at that volume.
False negatives (missed, but actually needed review): all 3 worst cases are page_3_5 rows with huge volume (9,800–22,000 impressions) and a genuinely terrible CTR (0.00–0.02%) — the model has learned "high impressions usually means fine," so it under-flags the rare huge page that's still doing badly.
By tier: accuracy is 77% on page_3_5, but only 69–71% on page_1 and striking — the model struggles more in the tiers where a "good" CTR and a "bad" one are closer together in absolute terms.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# what the model leans on
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 8 feature importances:")
print(imp.head(8))
print()

# concrete wrong cases
err = d.loc[test_mask, ["content_id", "position_tier", "avg_position", "ctr", "impressions_90d"]].reset_index(drop=True)
err["y_true"] = y_test
err["rf_prob"] = rf_probs
err["pred"] = (err["rf_prob"] >= 0.5).astype(int)

fp = err[(err["pred"] == 1) & (err["y_true"] == 0)].sort_values("rf_prob", ascending=False).head(3)
fn = err[(err["pred"] == 0) & (err["y_true"] == 1)].sort_values("rf_prob").head(3)
print("False positives (flagged, actually fine):")
print(fp)
print()
print("False negatives (missed, actually needed review):")
print(fn)

# accuracy by tier
err["correct"] = err["pred"] == err["y_true"]
print()
print("Accuracy by position_tier:")
print(err.groupby("position_tier")["correct"].mean().round(3))



Top 8 feature importances:
impressions_90d                 0.402038
days_with_impressions           0.268824
avg_position                    0.058377
content_age_days                0.038057
word_count                      0.037661
char_count                      0.034879
days_since_last_update          0.023375
content_type_keyword article    0.018481
dtype: float64

False positives (flagged, actually fine):
                content_id position_tier  avg_position    ctr  \
3063  content_bd54ada85755        page_1           7.3  12.50   
397   content_c73c63198a13        page_1           4.0  50.00   
1306  content_d62a7b3da232        page_1           5.9   9.09   

      impressions_90d  y_true   rf_prob  pred  
3063                8       0  0.953003     1  
397                 2       0  0.934950     1  
1306               11       0  0.930888     1  

False negatives (missed, actually needed review):
                content_id position_tier  avg_position   ctr  impressions_90d  \
14

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.